##### config

In [10]:
%run 00_config

StatementMeta(, 90341e2f-d3dd-4c11-bfbe-b752967187f3, 12, Finished, Available, Finished, True)

FHIR Pipeline Config loaded
  FHIR URL   : https://hapi.fhir.org/baseR4
  Resources  : ['Patient', 'Encounter', 'Observation', 'Condition']
  Ingest days: 3
  Schemas    : bronze | silver | gold
  Naming     : tables → tbl_  | views → vw_

Table names:
  bronze.tbl_patient                  → silver.tbl_patient                  → gold.vw_dim_patient
  bronze.tbl_encounter                → silver.tbl_encounter                → gold.vw_fact_encounter
  bronze.tbl_observation              → silver.tbl_observation              → gold.vw_fact_observation
  bronze.tbl_condition                → silver.tbl_condition                → gold.vw_fact_condition


In [11]:
from datetime import datetime

CURRENT_TS = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")

print(f"Creating gold views | timestamp: {CURRENT_TS}\n")

StatementMeta(, 90341e2f-d3dd-4c11-bfbe-b752967187f3, 13, Finished, Available, Finished, False)

Creating gold views | timestamp: 2026-06-16T17:18:21



##### dim_patient

In [12]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_view('patient')} AS
SELECT
    resource_id        AS patient_id,
    gender,
    birth_date,
    deceased,
    effective_from     AS valid_from,
    is_current
FROM {silver_table('Patient')}
WHERE is_current = true
""")
print(f"Created: {gold_view('patient')}")

StatementMeta(, 90341e2f-d3dd-4c11-bfbe-b752967187f3, 14, Finished, Available, Finished, False)

Created: gold.vw_dim_patient


##### fact_encounter

In [13]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_view('encounter')} AS
SELECT
    resource_id        AS encounter_id,
    status,
    encounter_class,
    load_date          AS encounter_date,
    effective_from     AS valid_from
FROM {silver_table('Encounter')}
WHERE is_current = true
""")
print(f"Created: {gold_view('encounter')}")

StatementMeta(, 90341e2f-d3dd-4c11-bfbe-b752967187f3, 15, Finished, Available, Finished, False)

Created: gold.vw_fact_encounter


##### fact_observation

In [14]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_view('observation')} AS
SELECT
    resource_id        AS observation_id,
    status,
    effective_date,
    subject_ref,
    load_date,
    effective_from     AS valid_from
FROM {silver_table('Observation')}
WHERE is_current = true
""")
print(f"Created: {gold_view('observation')}")

StatementMeta(, 90341e2f-d3dd-4c11-bfbe-b752967187f3, 16, Finished, Available, Finished, False)

Created: gold.vw_fact_observation


##### fact_condition

In [15]:
spark.sql(f"""
CREATE OR REPLACE VIEW {gold_view('condition')} AS
SELECT
    resource_id        AS condition_id,
    clinical_status,
    verification_status,
    recorded_date,
    subject_ref,
    load_date,
    effective_from     AS valid_from
FROM {silver_table('Condition')}
WHERE is_current = true
""")
print(f"Created: {gold_view('condition')}")

print("\nAll gold views created!")

StatementMeta(, 90341e2f-d3dd-4c11-bfbe-b752967187f3, 17, Finished, Available, Finished, False)

Created: gold.vw_fact_condition

All gold views created!
